# Solution B2 · Missing values: native NaN in XGBoost, imputation and indicators

**Type**: Implement with scikit-learn

## Objective

Compare in repeated cross-validation three ways of handling missing values with XGBoost on the loans (where the missingness of the income is linked to the outcome): **native NaN**, median imputation, median plus missingness indicators; and with a random forest (which requires imputation). Measure the differences with their standard
error: many are small.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RepeatedStratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline

df = ds.loans(n=3000)
df["ratio"] = df["amount"] / df["income"]                        # NaN where the income is missing
X, y = df[["ratio", "age", "income", "job_tenure", "amount"]], df["defaulted"]
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)
print("share of missing values:", X.isna().mean().round(3).to_dict())


def xgb_(): return xgb.XGBClassifier(n_estimators=150, learning_rate=0.05, max_depth=3, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", n_jobs=-1, random_state=0)

## Commented solution

In [ ]:
models = {
    "xgb native": xgb_(),
    "xgb median": make_pipeline(SimpleImputer(strategy="median"), xgb_()),
    "xgb median+indicators": make_pipeline(SimpleImputer(strategy="median", add_indicator=True), xgb_()),
    "rf median+indicators": make_pipeline(SimpleImputer(strategy="median", add_indicator=True), RandomForestClassifier(200, min_samples_leaf=5, random_state=0, n_jobs=-1)),
}

In [ ]:
scores = {name: cross_val_score(m, X, y, cv=cv, scoring="roc_auc") for name, m in models.items()}
table = pd.DataFrame({n: {"auc": s.mean(), "sd": s.std()} for n, s in scores.items()}).T
table.round(4)

In [ ]:
rows = {}
for name in ["xgb median", "xgb median+indicators", "rf median+indicators"]:
    d = scores["xgb native"] - scores[name]                                  # paired comparison fold by fold
    rows[name] = {"diff_vs_native": d.mean(), "standard_error": d.std() / np.sqrt(len(d))}
diff = pd.DataFrame(rows).T
native_better_than_median = bool(diff.loc["xgb median", "diff_vs_native"] > 2 * diff.loc["xgb median", "standard_error"])
indicators_close_gap = bool(abs(diff.loc["xgb median+indicators", "diff_vs_native"]) <= diff.loc["xgb median+indicators", "standard_error"])
print(diff.round(4))
print("native better than the median alone:", native_better_than_median, "| with the indicators the difference disappears:", indicators_close_gap)

## Check

In [ ]:
with verify():
    check(list(models) == ["xgb native", "xgb median", "xgb median+indicators", "rf median+indicators"] and len(scores) == 4 and all(len(s) == 20 for s in scores.values()), "four models, twenty scores each", "cross_val_score with RepeatedStratifiedKFold(5, 4)")
    check(table["auc"].min() > 0.72 and table["auc"].max() < 0.86, "AUC between 0.72 and 0.86 for all", "the problem has a realistic AUC")
    check(list(diff.index) == ["xgb median", "xgb median+indicators", "rf median+indicators"] and list(diff.columns) == ["diff_vs_native", "standard_error"], "paired differences for the three comparisons", "sd of the differences / sqrt(20)")
    check(diff.loc["xgb median", "diff_vs_native"] > 0 and native_better_than_median is True, "native NaN beat the median alone (by over two standard errors)", "the missingness information is lost with imputation alone")
    check(indicators_close_gap is True and abs(diff.loc["xgb median+indicators", "diff_vs_native"]) < 0.01, "with the missingness indicator the difference from the native disappears (within one standard error, less than one point)", "median + add_indicator is equivalent to the native")
done()

## What you learned

- **XGBoost uses missingness as information:** at each split it chooses the direction in which to send the NaN that reduces the loss the most, so it learns that "missing income" indicates a higher default risk. With the median alone this information is **lost**: the AUC drops by about 0.6-1 points systematically (more than two standard errors).
- **The missingness indicator closes the gap:** median plus `add_indicator=True` makes the missingness explicit and brings the model back to the level of the native one (difference within one standard error). It is the result of module 4 (B1) seen from another model: the signal is in **the fact that it is missing**.
- **The differences are small:** less than one point of AUC, with a standard deviation across folds of 2-3 points. The paired comparison (fold-by-fold difference, with its standard error) is the way to see effects this small: a comparison between two means would not distinguish them from noise.
- The random forest (which does not accept NaN) requires imputation and with the indicators reaches comparable levels: the choice of XGBoost for missing values is a **convenience** with a small advantage, not a quality leap.

## Common pitfall

Imputing "to be safe" the data of a model that handles NaN: if the missingness is informative (data not missing at random) imputation without an indicator erases it. Try native and imputation+indicators and compare in cross-validation with the standard error.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How does XGBoost handle missing values?</summary>

At each split it learns a default direction for the NaN (left or right) choosing the one that minimizes the loss on the training data; so the missing values become a signal and no imputation or indicators are needed. If the missingness is informative it is an advantage, but with other models you get the same effect with imputation plus missingness indicators;
I always compare the two routes in cross-validation.

</details>